# 📦 Step 1 — Build Unified Combined Dataset (Fast)

Combines all 5 signal files per participant into one row-per-sample dataset **at 32 Hz**.

### Why 32 Hz (not downsampling)?
Downsampling required a Python loop over 875K rows → slow.  
Instead we **upsample the slower signals** to match 32 Hz:

| Signal | Original rate | Strategy |
|---|---|---|
| Nasal Airflow | 32 Hz | master clock — use as-is |
| Thoracic Movement | 32 Hz | use as-is |
| SpO2 | 4 Hz | repeat each value **8×** (32/4) |
| Sleep profile | 1 per 30 sec | repeat each stage **960×** (32×30) |
| Flow Events | annotations | vectorised IntervalIndex lookup |

### Final columns
```
datetime | participant | sleep_stage | event_label | nasal | spo2 | thoracic
```

### Sleep stages handled
Wake, N1, N2, N3, N4, REM, Movement

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'pandas', 'numpy'], check=True)
print('✅ Dependencies ready')

In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
import time

# ╔══════════════════════════════════════════════╗
# ║  CONFIG                                     ║
# ╚══════════════════════════════════════════════╝
BASE_DIR = Path(r'D:\SRIP_Health')
DATA_DIR = BASE_DIR / 'Data'
OUT_DIR  = BASE_DIR / 'Dataset'
OUT_DIR.mkdir(exist_ok=True)

PIDS     = ['AP01', 'AP02', 'AP03', 'AP04', 'AP05']
FS       = 32          # master sample rate (Hz)
FS_SPO2  = 4           # SpO2 sample rate
EPOCH_SEC = 30         # sleep profile epoch length

SPO2_REPEAT   = FS // FS_SPO2          # 8  — repeat each SpO2 value 8×
STAGE_REPEAT  = FS * EPOCH_SEC         # 960 — repeat each stage epoch 960×

SLEEP_STAGES = {'Wake', 'N1', 'N2', 'N3', 'N4', 'REM', 'Movement'}

print(f'Master rate      : {FS} Hz')
print(f'SpO2 repeat      : {SPO2_REPEAT}×  (one SpO2 value per {SPO2_REPEAT} nasal samples)')
print(f'Stage repeat     : {STAGE_REPEAT}×  (one epoch per {STAGE_REPEAT} samples = 30 sec)')
print(f'Base dir         : {BASE_DIR}')

In [ ]:
# ══════════════════════════════════════════════
#  PARSERS
# ══════════════════════════════════════════════

def parse_ts(s: str) -> datetime:
    """'30.05.2024 20:59:00,119' → datetime (comma = millisecond separator)"""
    s = s.strip().replace(',', '.')
    for fmt in ('%d.%m.%Y %H:%M:%S.%f', '%d.%m.%Y %H:%M:%S'):
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            pass
    raise ValueError(f'Cannot parse: {s}')


def parse_signal_file(filepath: Path, value_col: str) -> pd.DataFrame:
    """
    Reads Flow / Thorac / SPO2 files.
    Skips metadata header, reads from 'Data:' line onward.
    Returns: DataFrame(datetime, value_col)
    """
    rows = []
    in_data = False
    with open(filepath, encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.strip()
            if line == 'Data:':
                in_data = True
                continue
            if not in_data or not line:
                continue
            parts = line.split(';')
            if len(parts) < 2:
                continue
            try:
                rows.append((parse_ts(parts[0]), float(parts[1].strip())))
            except Exception:
                continue
    df = pd.DataFrame(rows, columns=['datetime', value_col])
    print(f'  {filepath.name:45s} → {len(df):>8,} rows')
    return df


EV_RE = re.compile(
    r'^(\d{2}\.\d{2}\.\d{4})\s+([\d:,]+)-([\d:,]+);\s*([\d.]+);\s*([^;\n]+)(?:;\s*([^;\n]*))?'
)

def parse_events_file(filepath: Path) -> pd.DataFrame:
    """Returns: DataFrame(start_dt, end_dt, duration_sec, event_label, sleep_stage)"""
    rows = []
    with open(filepath, encoding='utf-8', errors='replace') as f:
        for line in f:
            m = EV_RE.match(line.strip())
            if not m:
                continue
            date_str, t_start, t_end, dur, label, stage = m.groups()
            try:
                s = parse_ts(f'{date_str} {t_start}')
                e = parse_ts(f'{date_str} {t_end}')
                if e < s:               # midnight crossover
                    e += timedelta(days=1)
                rows.append({
                    'start_dt':     s,
                    'end_dt':       e,
                    'duration_sec': float(dur),
                    'event_label':  label.strip(),
                    'sleep_stage':  (stage or '').strip(),
                })
            except Exception:
                continue
    df = pd.DataFrame(rows)
    print(f'  {filepath.name:45s} → {len(df):>8,} events')
    return df


SP_RE = re.compile(r'^(\d{2}\.\d{2}\.\d{4}\s+[\d:,]+);\s*(.+)$')

def parse_sleep_profile(filepath: Path) -> pd.DataFrame:
    """Returns: DataFrame(datetime, sleep_stage) — one row per 30-sec epoch"""
    rows = []
    with open(filepath, encoding='utf-8', errors='replace') as f:
        for line in f:
            m = SP_RE.match(line.strip())
            if not m:
                continue
            stage = m.group(2).strip()
            if stage not in SLEEP_STAGES:
                continue
            try:
                rows.append({
                    'datetime':    parse_ts(m.group(1)),
                    'sleep_stage': stage,
                })
            except Exception:
                continue
    df = pd.DataFrame(rows)
    print(f'  {filepath.name:45s} → {len(df):>8,} epochs  '
          f'({len(df) * EPOCH_SEC / 3600:.1f} h)')
    return df


def get_file(pid: str, prefix: str) -> Path:
    folder  = DATA_DIR / pid
    matches = list(folder.glob(f'{prefix}*'))
    assert len(matches) == 1, (
        f'Expected 1 file matching "{prefix}*" in {folder}, '
        f'found {len(matches)}: {matches}'
    )
    return matches[0]


print('✅ All parsers defined')

In [ ]:
# ══════════════════════════════════════════════
#  FAST UPSAMPLING HELPERS
# ══════════════════════════════════════════════

def upsample_spo2(spo2_df: pd.DataFrame, n_nasal: int) -> np.ndarray:
    """
    Repeats each SpO2 value SPO2_REPEAT (8) times to match 32 Hz.

    e.g. [93, 94, 94] at 4 Hz  →  [93,93,93,93,93,93,93,93, 94,94,...] at 32 Hz

    If the repeat doesn't exactly match n_nasal (edge at end of recording),
    we truncate or pad with the last value.
    """
    arr = np.repeat(spo2_df['spo2'].values, SPO2_REPEAT)
    # Align length to nasal signal
    if len(arr) >= n_nasal:
        return arr[:n_nasal]
    else:
        pad = np.full(n_nasal - len(arr), arr[-1])
        return np.concatenate([arr, pad])


def upsample_sleep_profile(sleep_df: pd.DataFrame,
                            nasal_datetimes: pd.Series) -> np.ndarray:
    """
    For each nasal sample timestamp, finds the last sleep epoch
    that started at or before it — vectorised with searchsorted.

    This is O(n log m) rather than O(n*m).
    n = 875K nasal samples, m = 912 sleep epochs.
    """
    sp       = sleep_df.sort_values('datetime').reset_index(drop=True)
    ep_times = sp['datetime'].values          # numpy datetime64 array
    ep_stages = sp['sleep_stage'].values

    sample_times = nasal_datetimes.values     # numpy datetime64 array

    # searchsorted: for each sample time, find the insertion point
    # in ep_times — then subtract 1 to get the last epoch <= sample time
    idx = np.searchsorted(ep_times, sample_times, side='right') - 1

    result = np.full(len(sample_times), 'Unknown', dtype=object)
    valid  = idx >= 0
    result[valid] = ep_stages[idx[valid]]
    # Samples before the first epoch get idx=-1 → Unknown
    # Fix: assign them the first known stage (backward fill)
    if not valid.all():
        first_known = ep_stages[0] if len(ep_stages) > 0 else 'Wake'
        result[~valid] = first_known
    return result


def label_events_fast(nasal_datetimes: pd.Series,
                       events_df: pd.DataFrame) -> np.ndarray:
    """
    Labels each sample as its event type or 'Normal'.

    Uses pd.IntervalIndex for vectorised interval lookup.
    O(n log m) — no Python loop over 875K rows.

    For overlapping events (rare), the first event wins.
    """
    if events_df.empty:
        return np.full(len(nasal_datetimes), 'Normal', dtype=object)

    # Build IntervalIndex from event start/end datetimes
    iv_index = pd.IntervalIndex.from_arrays(
        events_df['start_dt'].values,
        events_df['end_dt'].values,
        closed='both'
    )
    labels_arr = events_df['event_label'].values

    sample_times = nasal_datetimes.values
    result = np.full(len(sample_times), 'Normal', dtype=object)

    # get_indexer returns -1 for no match, index into iv_index otherwise
    # Note: get_indexer doesn't handle overlapping intervals — we use
    # get_loc in a vectorised way via get_indexer
    # get_indexer_non_unique returns (indexer, missing) — must unpack
    hits, _ = iv_index.get_indexer_non_unique(sample_times)
    matched = hits >= 0
    result[matched] = labels_arr[hits[matched]]

    return result


print('✅ upsample_spo2()          — repeat each SpO2 value 8×')
print('✅ upsample_sleep_profile() — searchsorted, O(n log m)')
print('✅ label_events_fast()      — IntervalIndex, vectorised')

In [ ]:
# ══════════════════════════════════════════════
#  MAIN LOOP
# ══════════════════════════════════════════════

all_parts = []
t_total   = time.time()

for pid in PIDS:
    print(f'\n{"═"*58}')
    print(f'  {pid}')
    print(f'{"═"*58}')
    t0 = time.time()

    # ── 1. Parse ─────────────────────────────────────────
    nasal_df  = parse_signal_file(get_file(pid, 'Flow -'),          'nasal')
    thorac_df = parse_signal_file(get_file(pid, 'Thorac -'),        'thoracic')
    spo2_df   = parse_signal_file(get_file(pid, 'SPO2 -'),          'spo2')
    events_df = parse_events_file(get_file(pid, 'Flow Events -'))
    sleep_df  = parse_sleep_profile(get_file(pid, 'Sleep profile -'))

    n = len(nasal_df)   # master length
    print(f'  Master length  : {n:,} samples  ({n / FS / 3600:.2f} h at {FS} Hz)')

    # ── 2. Align thoracic to nasal (both 32 Hz, same length) ─
    # Truncate/pad if there is a tiny mismatch (rare)
    thorac_vals = thorac_df['thoracic'].values
    if len(thorac_vals) >= n:
        thorac_vals = thorac_vals[:n]
    else:
        thorac_vals = np.concatenate([
            thorac_vals,
            np.full(n - len(thorac_vals), thorac_vals[-1])
        ])

    # ── 3. Upsample SpO2: repeat each value 8× ────────────
    print('  Upsampling SpO2 (repeat 8×)...')
    spo2_vals = upsample_spo2(spo2_df, n)
    print(f'  SpO2 upsampled : {len(spo2_vals):,} values')

    # ── 4. Upsample sleep profile: searchsorted ────────────
    print('  Upsampling sleep profile (searchsorted)...')
    t1 = time.time()
    stage_vals = upsample_sleep_profile(sleep_df, nasal_df['datetime'])
    print(f'  Sleep stage    : done in {time.time()-t1:.1f}s')

    # ── 5. Label events: IntervalIndex ────────────────────
    print('  Labelling events (IntervalIndex)...')
    t2 = time.time()
    event_vals = label_events_fast(nasal_df['datetime'], events_df)
    n_event    = (event_vals != 'Normal').sum()
    print(f'  Events labelled: done in {time.time()-t2:.1f}s  '
          f'({n_event:,} non-Normal samples)')

    # ── 6. Build combined DataFrame ───────────────────────
    combined = pd.DataFrame({
        'datetime':    nasal_df['datetime'],
        'participant': pid,
        'sleep_stage': stage_vals,
        'event_label': event_vals,
        'nasal':       nasal_df['nasal'].values,
        'spo2':        spo2_vals,
        'thoracic':    thorac_vals,
    })

    # ── 7. Stats ──────────────────────────────────────────
    print(f'  Total time     : {time.time()-t0:.1f}s')
    print(f'  Event counts   :')
    for lbl, cnt in combined['event_label'].value_counts().items():
        print(f'    {lbl:25s} {cnt:>8,}  ({cnt/n*100:.2f}%)')
    print(f'  Stage counts   :')
    for lbl, cnt in combined['sleep_stage'].value_counts().items():
        print(f'    {lbl:25s} {cnt:>8,}  ({cnt/n*100:.2f}%)')

    all_parts.append(combined)

# ── Stack all participants ──────────────────────────────────
print(f'\n{"═"*58}')
print('  Combining all participants...')
final_df = pd.concat(all_parts, ignore_index=True)

out_path = OUT_DIR / 'combined_dataset.csv'
print(f'  Saving → {out_path}')
final_df.to_csv(out_path, index=False)

print(f'\n✅ Done in {time.time()-t_total:.1f}s total')
print(f'   Rows    : {len(final_df):,}')
print(f'   Columns : {list(final_df.columns)}')
print(f'\nOverall event distribution:')
print(final_df['event_label'].value_counts().to_string())
print(f'\nOverall sleep stage distribution:')
print(final_df['sleep_stage'].value_counts().to_string())

In [ ]:
# ── Sanity check ───────────────────────────────────────────
print('First 5 rows:')
print(final_df.head().to_string(index=False))
print()
print('5 rows from an event window:')
ev_rows = final_df[final_df['event_label'] != 'Normal'].head(5)
print(ev_rows.to_string(index=False))
print()
print('5 rows where sleep_stage = N4 (if present):')
n4 = final_df[final_df['sleep_stage'] == 'N4']
if len(n4):
    print(n4.head(5).to_string(index=False))
else:
    print('  No N4 epochs found in this dataset')